# Submission v12 — v7c features + within-session LAG features ONLY (isolated)

## Single isolated change vs v7c
- Same v7c hyperparameters (`num_leaves=127, n_estimators=1000, lr=0.02, subsample=0.6, reg 0.3`, early stopping)
- Same v7c features (106 features)
- Same class weights (capped at 2.5)
- Same 3 seeds × 5 folds
- Same calibration α=1.8

**Only addition:** within-session lag features. For each label, look up the previous label's sensor stats from the same nurse, but ONLY if previous label is within 600s (10min) — otherwise different session.

NO SMOTE, NO CatBoost. Pure isolation of lag effect.

## Sanity check + decision rule
- If LGB-only LOPO >= 0.5130 → lag features work, decide based on margin
- If LGB-only LOPO < 0.5130 → lag features hurt (likely cross-fold leakage), discard

## Sanity safeguard
We also save a version WITHOUT lag features as a sanity baseline. If that doesn't reproduce v7c's 0.5130, the entire run is invalid.


In [1]:
%pip install lightgbm scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS = 180_000
HALF_MS   =  90_000
THIRD_MS  =  60_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']:
            f['hrv_'+k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn']/f['hrv_mean_rr'] if f['hrv_mean_rr']>1e-6 else 0.
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]
        for col in SENSOR_COLS:
            v   = wa[col].dropna().values.astype(float)
            vf  = wf[col].dropna().values.astype(float)
            vl  = wl[col].dropna().values.astype(float)
            vt1 = wt1[col].dropna().values.astype(float)
            vt3 = wt3[col].dropna().values.astype(float)
            if len(v)==0:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr',
                          'delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[col+'_'+s] = np.nan
                continue
            feat[col+'_mean']   = float(np.mean(v))
            feat[col+'_std']    = float(np.std(v))
            feat[col+'_min']    = float(np.min(v))
            feat[col+'_max']    = float(np.max(v))
            feat[col+'_median'] = float(np.median(v))
            feat[col+'_skew']   = float(spstats.skew(v)) if len(v)>2 else 0.
            feat[col+'_kurt']   = float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[col+'_range']  = float(np.max(v)-np.min(v))
            feat[col+'_q25']    = float(np.percentile(v,25))
            feat[col+'_q75']    = float(np.percentile(v,75))
            feat[col+'_iqr']    = float(np.percentile(v,75)-np.percentile(v,25))
            feat[col+'_delta']  = float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[col+'_slope']  = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.
            feat[col+'_t1_mean'] = float(np.mean(vt1)) if len(vt1)>0 else 0.
            feat[col+'_t3_mean'] = float(np.mean(vt3)) if len(vt3)>0 else 0.
            feat[col+'_t3t1']    = feat[col+'_t3_mean']-feat[col+'_t1_mean']

        if len(wa) > 0:
            ax = wa['accel_x'].fillna(0).values; ay = wa['accel_y'].fillna(0).values; az = wa['accel_z'].fillna(0).values
            mag = np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan

        feat.update(hrv_time_domain(wa['heart_rate']))
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p:i for i,p in enumerate(TRAIN_LABEL['pid'].unique())}

print('Extracting train features (v7c base)...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print(f'  train shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, train_pid_map)
print(f'  test shape: {test_features.shape}')


Extracting train features (v7c base)...
  train shape: (815, 106)
Extracting test features...
  test shape: (1028, 106)


## Add within-session lag features

For each label `id`, find the previous label from the same `pid` whose timestamp is within 600s (10min). Pull that previous window's sensor stats (mean/std/delta) for the 3 most informative sensors (eda, heart_rate, temperature) — 9 lag features + 1 availability flag = 10 new features.

Why these 3 sensors only: lag features should capture stress trajectory, not movement. Accelerometer lag is noise.

In [4]:
LAG_LIMIT_MS = 600_000  # 10 minutes
LAG_SENSORS  = ['eda','heart_rate','temperature']
LAG_STATS    = ['mean','std','delta']

def add_lag_features(features_df, label_df):
    """Add prev-window stats from same nurse if within session gap."""
    lab = label_df.copy().reset_index(drop=True)
    lab['timestamp'] = lab['timestamp'].astype(float)
    lab = lab.sort_values(['pid','timestamp']).reset_index(drop=True)

    prev_id_for = {}
    for _, grp in lab.groupby('pid'):
        ids = grp['id'].values
        ts  = grp['timestamp'].values
        for i in range(1, len(ids)):
            if (ts[i] - ts[i-1]) <= LAG_LIMIT_MS:
                prev_id_for[ids[i]] = ids[i-1]

    out = features_df.copy()
    avail_flag = []
    for s in LAG_SENSORS:
        for st in LAG_STATS:
            out[f'lag_{s}_{st}'] = np.nan
    for lid in out.index:
        prev = prev_id_for.get(lid, None)
        if prev is not None and prev in features_df.index:
            avail_flag.append(1)
            for s in LAG_SENSORS:
                for st in LAG_STATS:
                    col = f'{s}_{st}'
                    if col in features_df.columns:
                        out.at[lid, f'lag_{s}_{st}'] = features_df.at[prev, col]
        else:
            avail_flag.append(0)
    out['lag_avail'] = avail_flag
    return out

print('Adding lag features (within-session, ≤10min gap)...')
train_features_lag = add_lag_features(train_features, TRAIN_LABEL)
test_features_lag  = add_lag_features(test_features,  TEST_LABEL)

print(f'  train shape (with lag): {train_features_lag.shape}')
print(f'  test shape  (with lag): {test_features_lag.shape}')
print(f'  lag_avail rate train: {train_features_lag["lag_avail"].mean()*100:.1f}%')
print(f'  lag_avail rate test:  {test_features_lag["lag_avail"].mean()*100:.1f}%')


Adding lag features (within-session, ≤10min gap)...
  train shape (with lag): (815, 116)
  test shape  (with lag): (1028, 116)
  lag_avail rate train: 90.7%
  lag_avail rate test:  87.8%


In [5]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

# Build TWO sets: one without lag (sanity baseline), one with lag (test)
imputer_base = SimpleImputer(strategy='median')
X_base       = pd.DataFrame(imputer_base.fit_transform(train_features),
                             columns=train_features.columns, index=train_features.index)
X_test_base  = pd.DataFrame(imputer_base.transform(test_features),
                             columns=test_features.columns, index=test_features.index)

imputer_lag = SimpleImputer(strategy='median')
X_lag       = pd.DataFrame(imputer_lag.fit_transform(train_features_lag),
                            columns=train_features_lag.columns, index=train_features_lag.index)
X_test_lag  = pd.DataFrame(imputer_lag.transform(test_features_lag),
                            columns=test_features_lag.columns, index=test_features_lag.index)

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {0: total/(n_cls*counts[0]),
                 1: min(total/(n_cls*counts[1]), 2.5),
                 2: total/(n_cls*counts[2])}
sample_weights = np.array([class_weights[yi] for yi in y])
train_prior    = np.array([counts[i]/total for i in range(3)])

print('X_base shape (no lag):', X_base.shape)
print('X_lag  shape (with lag):', X_lag.shape)
print('Class weights (capped):', {k: round(v,3) for k,v in class_weights.items()})


X_base shape (no lag): (815, 106)
X_lag  shape (with lag): (815, 116)
Class weights (capped): {0: 1.677, 1: 2.5, 2: 0.463}


## LOPO CV — baseline sanity + lag test

Run LOPO twice: once on X_base (no lag, must reproduce v7c=0.5130), once on X_lag.
This tests EXACTLY the effect of adding lag features.

In [6]:
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.02,
    num_leaves        = 127,
    max_depth         = -1,
    min_child_samples = 5,
    subsample         = 0.6,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.3,
    reg_lambda        = 0.3,
    class_weight      = 'balanced',
    objective         = 'multiclass',
    num_class         = 3,
    n_jobs            = -1,
    verbose           = -1,
)

def run_lopo(X, label):
    logo = LeaveOneGroupOut()
    scores = []
    print(f'=== LOPO CV — {label} ===')
    for tr_idx, va_idx in logo.split(X, y, groups):
        pid_val = groups.iloc[va_idx[0]]
        y_tr = y.iloc[tr_idx]
        y_va = y.iloc[va_idx]
        if len(y_va.unique()) < 2:
            print(f'  Skip {pid_val}: only one class')
            continue
        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
        m.fit(X.iloc[tr_idx], y_tr,
              sample_weight=sample_weights[tr_idx],
              eval_set=[(X.iloc[va_idx], y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        sc = balanced_accuracy_score(y_va, m.predict(X.iloc[va_idx]))
        print(f'  Leave out {pid_val}: {sc:.4f}  (n={len(va_idx)})')
        scores.append(sc)
    return np.mean(scores), np.std(scores)

mu_base, sd_base = run_lopo(X_base, 'v12 baseline (NO lag) — sanity')
print()
mu_lag,  sd_lag  = run_lopo(X_lag,  'v12 +lag features')
print()
print(f'Baseline (no lag) LOPO = {mu_base:.4f} +/- {sd_base:.4f}')
print(f'+lag features    LOPO = {mu_lag:.4f} +/- {sd_lag:.4f}')
print(f'v7c reference         = 0.5130')
print()
sanity_ok = abs(mu_base - 0.5130) < 0.01
print(f'Sanity check (baseline ≈ 0.5130): {"PASS" if sanity_ok else "FAIL"}')
print(f'Lag effect: {mu_lag - mu_base:+.4f}')


=== LOPO CV — v12 baseline (NO lag) — sanity ===
  Leave out 43JW: 0.5000  (n=93)
  Leave out C8Q6: 0.4965  (n=152)
  Leave out DT5C: 0.5460  (n=90)
  Leave out F1ZM: 0.5000  (n=137)
  Leave out HDS9: 0.6517  (n=135)
  Leave out P4DZ: 0.2826  (n=144)
  Leave out TPQI: 0.6141  (n=64)

=== LOPO CV — v12 +lag features ===
  Leave out 43JW: 0.5000  (n=93)
  Leave out C8Q6: 0.4930  (n=152)
  Leave out DT5C: 0.5464  (n=90)
  Leave out F1ZM: 0.5000  (n=137)
  Leave out HDS9: 0.6154  (n=135)
  Leave out P4DZ: 0.2988  (n=144)
  Leave out TPQI: 0.6141  (n=64)

Baseline (no lag) LOPO = 0.5130 +/- 0.1097
+lag features    LOPO = 0.5097 +/- 0.0988
v7c reference         = 0.5130

Sanity check (baseline ≈ 0.5130): PASS
Lag effect: -0.0033


## Final ensemble — only the lag version

Build full submission with lag features.

In [7]:
SEEDS = [42, 7, 123]

all_test_proba = []
print('=== Final ensemble: 3 seeds x 5 folds (v7c params + lag features) ===')
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_scores, fold_test = [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lag, y), 1):
        X_tr = X_lag.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]
        X_va = X_lag.iloc[va_idx]
        y_va = y.iloc[va_idx]
        sw_tr = sample_weights[tr_idx]

        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

        sc = balanced_accuracy_score(y_va, m.predict(X_va))
        fold_scores.append(sc)
        print(f'  Seed {seed} Fold {fold}: val BA = {sc:.4f}')
        fold_test.append(m.predict_proba(X_test_lag))
    all_test_proba.append(np.mean(fold_test, axis=0))
    print(f'  Seed {seed} mean CV = {np.mean(fold_scores):.4f}')

raw_proba = np.mean(all_test_proba, axis=0)
print()
print('Raw distribution:')
pred = np.argmax(raw_proba, axis=1)
print(f'  ', dict(Counter(pred)))


=== Final ensemble: 3 seeds x 5 folds (v7c params + lag features) ===
  Seed 42 Fold 1: val BA = 0.8223
  Seed 42 Fold 2: val BA = 0.7892
  Seed 42 Fold 3: val BA = 0.8184
  Seed 42 Fold 4: val BA = 0.7678
  Seed 42 Fold 5: val BA = 0.8495
  Seed 42 mean CV = 0.8095
  Seed 7 Fold 1: val BA = 0.8283
  Seed 7 Fold 2: val BA = 0.8122
  Seed 7 Fold 3: val BA = 0.8298
  Seed 7 Fold 4: val BA = 0.8284
  Seed 7 Fold 5: val BA = 0.7698
  Seed 7 mean CV = 0.8137
  Seed 123 Fold 1: val BA = 0.8661
  Seed 123 Fold 2: val BA = 0.6553
  Seed 123 Fold 3: val BA = 0.8780
  Seed 123 Fold 4: val BA = 0.8220
  Seed 123 Fold 5: val BA = 0.8008
  Seed 123 mean CV = 0.8044

Raw distribution:
   {np.int64(2): 135, np.int64(0): 319, np.int64(1): 574}


In [8]:
CALIB_ALPHA = 1.8
cal = raw_proba * (train_prior ** CALIB_ALPHA)
cal = cal / cal.sum(axis=1, keepdims=True)
preds = np.argmax(cal, axis=1).astype(int)

submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds})
submission.to_csv('submission_v12_lag.csv', index=False)

print('=== Calibrated submission (alpha=1.8) ===')
print('  submission_v12_lag.csv')
print(f'  dist: {dict(Counter(preds))}')
print(f'  train prior: 0={counts[0]/total*100:.1f}%  1={counts[1]/total*100:.1f}%  2={counts[2]/total*100:.1f}%')
print()
print(f'=== Decision summary ===')
print(f'  Sanity baseline LOPO = {mu_base:.4f}  ({"PASS" if sanity_ok else "FAIL"})')
print(f'  Lag-version    LOPO  = {mu_lag:.4f}')
print(f'  Effect of lag        = {mu_lag - mu_base:+.4f}')
print(f'  v7c reference        = 0.5130')
print()
if not sanity_ok:
    print('>> Sanity FAILED — invalid run, do not submit')
elif mu_lag > 0.5130:
    print(f'>> Lag features WORK (+{mu_lag-0.5130:.4f}). Submit submission_v12_lag.csv')
else:
    print(f'>> Lag features do NOT help. Try SMOTE next or accept v7c.')


=== Calibrated submission (alpha=1.8) ===
  submission_v12_lag.csv
  dist: {np.int64(2): 842, np.int64(0): 153, np.int64(1): 33}
  train prior: 0=19.9%  1=8.1%  2=72.0%

=== Decision summary ===
  Sanity baseline LOPO = 0.5130  (PASS)
  Lag-version    LOPO  = 0.5097
  Effect of lag        = -0.0033
  v7c reference        = 0.5130

>> Lag features do NOT help. Try SMOTE next or accept v7c.
